# Notebook 12 -- Three-Angle Evaluation on RACE
## SLM-to-SLM Guided Reasoning Pipeline

**Why RACE?**

RACE (ReAding Comprehension from Examinations) contains English reading
comprehension questions from Chinese middle and high school exams.
Each question comes with a passage, and answering correctly requires:
- Understanding the passage content
- Inferring information not stated explicitly
- Applying logical reasoning across multiple sentences

This is fundamentally different from all previous datasets:
- Unlike math: no numeric computation
- Unlike science: requires reading a passage, not recalling facts
- Unlike commonsense: answer must be grounded in the given text

The guide must produce a plan that identifies WHICH part of the passage
is relevant and WHAT inference is needed — testing a new kind of reasoning.

**Pipeline Under Test:**
- Guide  : Qwen 2.5-3B-Instruct + LoRA fine-tuned adapter (1 forward pass)
- Solver : Qwen 2.5-1.5B-Instruct × 5 majority-vote passes
- Baseline: Qwen 2.5-1.5B-Instruct × 5 majority-vote passes (no guide plan)
- Random chance: **25.0%** (4 options: A, B, C, D)

**Dataset config:** We use the "high" subset (high school level) for
harder, more inference-heavy questions. The "middle" subset is easier
and more fact-retrieval focused.

**Three Evaluation Angles:**
1. Compute Efficiency  -- accuracy per billion parameter-passes
2. Vote Consistency    -- how reliably the ensemble agrees on the correct answer
3. Confidence Calibration -- how well confidence predicts correctness (ECE)


In [1]:
# CELL 1 -- Install (uncomment on first run)
# !pip install -q transformers==4.44.0
# !pip install -q peft==0.12.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")


Done.


In [2]:
# CELL 2 -- HuggingFace login
from huggingface_hub import login
login("")
print("HuggingFace login done")


HuggingFace login done


In [3]:
# CELL 3 -- Imports + GPU
import os, json, re, glob, random, time
import torch
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/race_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Output  : {OUTPUT_DIR}")


PyTorch : 2.10.0+cu128
GPU     : Tesla T4
VRAM    : 15.6 GB
Output  : /kaggle/working/race_eval


In [4]:
# CELL 4 -- Configuration
CONFIG = {
    # Models
    "guide_base"          : "Qwen/Qwen2.5-3B-Instruct",
    "response_model"      : "Qwen/Qwen2.5-1.5B-Instruct",

    # Dataset
    "dataset_name"        : "ehovy/race",
    "dataset_config"      : "high",       # high school level -- harder, more inference-heavy
    "dataset_split"       : "test",
    "max_eval_samples"    : 500,
    "random_seed"         : 42,           # FIXED -- same seed for BOTH conditions

    # Ensemble
    "n_votes"             : 5,
    "vote_temperature"    : 0.4,
    "guide_temperature"   : 0.1,
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 512,          # more tokens -- passage + question is longer

    # Compute cost (billions of parameters)
    "guide_params_B"      : 3.0,
    "solver_params_B"     : 1.5,

    # Paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")


Config ready:
  guide_base              : Qwen/Qwen2.5-3B-Instruct
  response_model          : Qwen/Qwen2.5-1.5B-Instruct
  dataset_name            : ehovy/race
  dataset_config          : high
  dataset_split           : test
  max_eval_samples        : 500
  random_seed             : 42
  n_votes                 : 5
  vote_temperature        : 0.4
  guide_temperature       : 0.1
  refiner_temperature     : 0.3
  max_new_tokens          : 512
  guide_params_B          : 3.0
  solver_params_B         : 1.5
  results_file            : /kaggle/working/race_eval/results.jsonl
  report_file             : /kaggle/working/race_eval/eval_report.json
  angle1_file             : /kaggle/working/race_eval/angle1_compute_efficiency.json
  angle2_file             : /kaggle/working/race_eval/angle2_vote_consistency.json
  angle3_file             : /kaggle/working/race_eval/angle3_confidence_calibration.json
  checkpoint_file         : /kaggle/working/race_eval/checkpoint.json
  save_every          

In [5]:
# CELL 5 -- Load RACE dataset
# RACE fields:
#   example_id  : str
#   article     : str  (the reading passage)
#   answer      : str  (single letter: 'A', 'B', 'C', or 'D')
#   question    : str
#   options     : list of 4 strings (the answer choices, already plain text)
#
# We format the input as:
#   Passage: <article>
#   Question: <question>
#   Options:
#   A) ...
#   B) ...
#   C) ...
#   D) ...
#
# NOTE: passages can be long. We truncate to 600 chars to stay within
# model context limits while preserving enough for inference.

PASSAGE_MAX_CHARS = 600  # truncate long passages for context window safety

print("Loading RACE (high) from HuggingFace...")
raw_ds = load_dataset(CONFIG["dataset_name"], CONFIG["dataset_config"])

print(f"Splits   : {list(raw_ds.keys())}")
print(f"Test size: {len(raw_ds[CONFIG['dataset_split']])}")

ex = raw_ds[CONFIG["dataset_split"]][0]
print(f"\nExample record keys: {list(ex.keys())}")
print(f"Answer: {ex['answer']}")
print(f"Question: {ex['question'][:100]}")
print(f"Options: {ex['options']}")

VALID_LETTERS = set("ABCD")

def normalise_race(item):
    """Convert RACE record to {question, answer} pipeline format."""
    article  = item["article"].strip()
    # Truncate very long passages
    if len(article) > PASSAGE_MAX_CHARS:
        article = article[:PASSAGE_MAX_CHARS].rsplit(" ", 1)[0] + "..."

    opts = item["options"]   # list of 4 strings
    options_str = "\n".join(f"{chr(65+i)}) {opt}" for i, opt in enumerate(opts))

    q = (
        f"Passage:\n{article}\n\n"
        f"Question: {item['question'].strip()}\n\n"
        f"Options:\n{options_str}"
    )
    ans = str(item["answer"]).strip().upper()
    # Occasionally answer is '1','2','3','4' -- normalise
    num_map = {"1":"A","2":"B","3":"C","4":"D"}
    ans = num_map.get(ans, ans)
    return {"question": q, "answer": ans}


all_data = [normalise_race(x) for x in raw_ds[CONFIG["dataset_split"]]]

# CRITICAL: fix seed ONCE before sampling
random.seed(CONFIG["random_seed"])
if CONFIG["max_eval_samples"] < len(all_data):
    test_data = random.sample(all_data, CONFIG["max_eval_samples"])
    print(f"\nSampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
else:
    test_data = all_data
    print(f"\nUsing all {len(test_data)} questions")

print(f"\nSample formatted input (first 400 chars):\n{test_data[0]['question'][:400]}")
print(f"\nAnswer: {test_data[0]['answer']}")


Loading RACE (high) from HuggingFace...


README.md: 0.00B [00:00, ?B/s]

high/test-00000-of-00001.parquet:   0%|          | 0.00/1.68M [00:00<?, ?B/s]

high/train-00000-of-00001.parquet:   0%|          | 0.00/30.4M [00:00<?, ?B/s]

high/validation-00000-of-00001.parquet:   0%|          | 0.00/1.66M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/3498 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/62445 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3451 [00:00<?, ? examples/s]

Splits   : ['test', 'train', 'validation']
Test size: 3498

Example record keys: ['example_id', 'article', 'answer', 'question', 'options']
Answer: C
Question: What did Nancy try to do before she fell over?
Options: ['Measure the depth of the river', 'Look for a fallen tree trunk', 'Protect her cows from being drowned', 'Run away from the flooded farm']

Sampled 500 questions (seed=42)

Sample formatted input (first 400 chars):
Passage:
In the dark and damp market,Chen ShuChu,near her sixties,owns a stall that her father left her.The stall,called YuanJin Vegetables,is her everything.Chen earns only a little money by selling vegetables,but she has donated about $321,550 to help poor children.
In March,Forbes magazine named her one of the 48 great philanthropists from the AsiaPacific region.A month later,TIME magazine sele

Answer: B


In [6]:
# CELL 6 -- Answer extraction for multiple-choice (A-D)
# RACE answers are single letters A, B, C, or D.

def extract_gt_answer(answer_str):
    """GT is already a clean letter."""
    s = str(answer_str).strip().upper()
    num_map = {"1":"A","2":"B","3":"C","4":"D"}
    s = num_map.get(s, s)
    return s if s in VALID_LETTERS else ""

def extract_pred_answer(text):
    """
    Extract chosen letter (A-D) from model free-form output.
    Priority: explicit answer phrases first, fallback to last standalone letter.
    """
    text = text.strip()

    # 1. Explicit answer phrases
    m = re.search(
        r"(?:the answer is|answer is|answer:|the correct answer is|correct answer is)"
        r"[\s:]*([A-D])\b",
        text, re.IGNORECASE
    )
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 2. "option/choice X is correct"
    m = re.search(
        r"(?:option|choice)\s+([A-D])\s+(?:is correct|is the answer|is right|matches)",
        text, re.IGNORECASE
    )
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 3. #### A marker
    m = re.search(r"####\s*([A-D])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 4. Parenthesised at end: (A)
    m = re.search(r"\(([A-D])\)\s*$", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 5. Bold: **A**
    m = re.search(r"\*\*([A-D])\)?\*\*", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 6. Standalone letter on its own line
    matches = re.findall(r"^\s*([A-D])\s*$", text, re.MULTILINE | re.IGNORECASE)
    if matches:
        return matches[-1].upper()

    # 7. Last standalone letter anywhere
    matches = re.findall(r"\b([A-D])\b", text, re.IGNORECASE)
    if matches:
        return matches[-1].upper()

    return ""

# Quick test
for t in ["The answer is C", "#### B", "(A)", "**D**", "answer: B"]:
    print(f"  '{t}' -> '{extract_pred_answer(t)}'")
print("Extraction OK")


  'The answer is C' -> 'C'
  '#### B' -> 'B'
  '(A)' -> 'A'
  '**D**' -> 'D'
  'answer: B' -> 'B'
Extraction OK


In [7]:
# CELL 7 -- Load fine-tuned guide model (Qwen 3B + LoRA)

def find_adapter():
    patterns = [
        "/kaggle/input/datasets/makkisakib1/final-adapter-qwen-svamp",
        "/kaggle/input/*/adapter",
        "/kaggle/input/*/final-adapter",
        "/kaggle/input/*/final_adapter",
    ]
    for p in patterns:
        for m in glob.glob(p):
            print(f"  Found adapter: {m}")
            return m
    return None

print(f"Loading guide base: {CONFIG['guide_base']}")
guide_tok = AutoTokenizer.from_pretrained(CONFIG["guide_base"], trust_remote_code=True)
guide_tok.padding_side = "left"
if guide_tok.pad_token is None:
    guide_tok.pad_token = guide_tok.eos_token

guide_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["guide_base"],
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

adapter_path = find_adapter()
if adapter_path:
    guide_model = PeftModel.from_pretrained(guide_model, adapter_path)
    print("LoRA adapter loaded -- fine-tuned guide active")
else:
    print("WARNING: No adapter found. Using base Qwen 3B as guide.")

guide_model.eval()
print(f"Guide VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Loading guide base: Qwen/Qwen2.5-3B-Instruct


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  Found adapter: /kaggle/input/datasets/makkisakib1/final-adapter-qwen-svamp
LoRA adapter loaded -- fine-tuned guide active
Guide VRAM: 3.14 GB


In [8]:
# CELL 8 -- Load solver model (Qwen 1.5B)

print(f"Loading solver: {CONFIG['response_model']}")
resp_tok = AutoTokenizer.from_pretrained(CONFIG["response_model"])
if resp_tok.pad_token is None:
    resp_tok.pad_token = resp_tok.eos_token

resp_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["response_model"],
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

total_vram = torch.cuda.memory_allocated() / 1e9
headroom   = 17.1 - total_vram
print(f"Total VRAM (both models): {total_vram:.2f} GB / 17.1 GB")
print(f"Headroom                : {headroom:.1f} GB")
print("Memory OK" if headroom >= 2 else "WARNING: Tight -- reduce n_votes to 3 if OOM")


Loading solver: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Total VRAM (both models): 4.83 GB / 17.1 GB
Headroom                : 12.3 GB
Memory OK


In [9]:
# CELL 9 -- Prompts and generation functions
# RACE-specific: guide reads the passage and identifies:
#   1. Which part of the passage is relevant to the question
#   2. What inference or reasoning connects passage to answer
#   3. Which option matches that reasoning

GUIDE_SYSTEM = (
    "You are a reading comprehension assistant for multiple-choice questions.\n"
    "You are given a passage and a question with options A-D.\n"
    "Write 2-3 concrete reasoning steps that lead to the correct answer.\n"
    "Each step must reference SPECIFIC text from the passage.\n"
    "Your LAST line must always be: Best answer: <letter> because <one-line reason>\n\n"
    "Rules:\n"
    "- Quote or paraphrase the EXACT part of the passage that supports the answer.\n"
    "- Explicitly eliminate at least one wrong option if possible.\n"
    "- Do not guess -- every step must be grounded in the passage text.\n"
    "- No markdown. Plain text only.\n\n"
    "BAD example (not grounded):\n"
    "  Step 1: The passage talks about the topic.\n"
    "  Step 2: Option C seems most relevant.\n"
    "  Best answer: C because it sounds right\n\n"
    "GOOD example (grounded in passage):\n"
    "  Step 1: The passage says 'the factory closed in 1987 due to falling demand'.\n"
    "           This directly answers why the factory closed -- economic reasons.\n"
    "  Step 2: Option A says 'fire' and option B says 'flood' -- neither is in the passage.\n"
    "           Option D says 'falling demand' -- matches the passage exactly.\n"
    "  Best answer: D because the passage explicitly states 'falling demand' as the cause\n\n"
    "Apply this pattern to any reading comprehension question -- inference, main idea, "
    "vocabulary in context, author purpose, or detail retrieval."
)

SOLVE_SYSTEM = (
    "You are a precise reading comprehension solver.\n"
    "You are given a passage, a question, and a reasoning plan.\n"
    "Follow the plan exactly. Pick the letter the plan identifies as correct.\n"
    "Do not contradict the plan or re-read the passage independently.\n"
    "No markdown. Plain text only.\n"
    "Your absolute last line must be exactly: The answer is [letter]\n\n"
    "Example:\n"
    "Plan says: Best answer: D because the passage explicitly states 'falling demand'.\n"
    "The answer is D"
)

SOLVE_BASELINE_SYSTEM = (
    "You are a reading comprehension question answering assistant.\n"
    "Read the passage carefully. Answer the question based only on the passage.\n"
    "Think step by step if needed.\n"
    "No markdown. Plain text only.\n"
    "Your absolute last line must be exactly: The answer is [letter]"
)

REFINER_SYSTEM = (
    "You are a careful reading comprehension checker.\n"
    "You are given a passage+question and a list of tied candidate answers.\n"
    "Re-read the passage and determine which answer is best supported by the text.\n"
    "Your absolute last line must be exactly: The answer is [letter]"
)


def _generate(model, tokenizer, system_prompt, user_prompt, temperature, max_new_tokens):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def generate_plan(question):
    return _generate(
        guide_model, guide_tok,
        GUIDE_SYSTEM, question,
        CONFIG["guide_temperature"], CONFIG["max_new_tokens"]
    )

def generate_guided(question, plan):
    prompt = f"Plan:\n{plan}\n\nNow answer:\n{question}"
    return _generate(
        resp_model, resp_tok,
        SOLVE_SYSTEM, prompt,
        CONFIG["vote_temperature"], CONFIG["max_new_tokens"]
    )

def generate_baseline(question):
    return _generate(
        resp_model, resp_tok,
        SOLVE_BASELINE_SYSTEM, question,
        CONFIG["vote_temperature"], CONFIG["max_new_tokens"]
    )

def generate_refiner(question, tied_answers):
    prompt = (
        f"Passage + Question:\n{question}\n\n"
        f"Tied candidate answers: {', '.join(tied_answers)}\n"
        f"Which is best supported by the passage?"
    )
    return _generate(
        resp_model, resp_tok,
        REFINER_SYSTEM, prompt,
        CONFIG["refiner_temperature"], CONFIG["max_new_tokens"]
    )

print("Prompts and generation functions ready")


Prompts and generation functions ready


In [10]:
# CELL 10 -- Voting logic with richer metrics

def vote_and_decide(answers, question, gt_answer=None):
    """Majority voting with refiner fallback on ties."""
    valid = [a for a in answers if a in VALID_LETTERS]
    if not valid:
        valid = answers

    vote_counts  = Counter(valid)
    most_common  = vote_counts.most_common()
    top_answer   = most_common[0][0]
    top_count    = most_common[0][1]
    total        = len(valid)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / max(total, 1)
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used    = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        ref_raw  = generate_refiner(question, list(valid))
        ref_ans  = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None

        all_v      = valid + ([ref_ans] if ref_ans in VALID_LETTERS else [])
        new_counts = Counter(all_v)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]

        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_v), 4)
        total      = len(all_v)
        correct_votes    = Counter(all_v).get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / max(total, 1)
        wasted     = total - new_top_c

    return {
        "final_answer"     : final,
        "strategy"         : strategy,
        "confidence"       : conf,
        "vote_counts"      : dict(vote_counts),
        "total_votes"      : total,
        "correct_votes"    : correct_votes,
        "vote_consistency" : round(vote_consistency, 4),
        "wasted_votes"     : wasted,
        "refiner_used"     : refiner_used,
        "refiner_correct"  : refiner_correct,
    }

print("Voting logic ready")


Voting logic ready


In [11]:
# CELL 11 -- Single question test (verify pipeline end-to-end)

print("=" * 65)
print("SINGLE QUESTION TEST  (RACE-High)")
print("=" * 65)

item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
print(f"Input (first 500 chars):\n{q[:500]}")
print(f"\nGT Answer: {gt}")

print("\n[1] Guide generating plan...")
plan = generate_plan(q)
print(f"Plan:\n{plan}")

print(f"\n[2] Guided votes ({CONFIG['n_votes']}x)...")
guided_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_guided(q, plan)
    pred = extract_pred_answer(raw)
    guided_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw snippet: {raw[:80]}")

g = vote_and_decide(guided_votes, q, gt)
print(f"\n  Result    : {g['final_answer']}  (GT: {gt})  {'CORRECT' if g['final_answer']==gt else 'WRONG'}")
print(f"  Strategy  : {g['strategy']}")
print(f"  Confidence: {g['confidence']}")
print(f"  Correct votes: {g['correct_votes']}/{g['total_votes']} ({g['vote_consistency']*100:.0f}%)")

print("\n[3] Baseline votes (no plan)...")
base_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_baseline(q)
    pred = extract_pred_answer(raw)
    base_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw snippet: {raw[:80]}")

b = vote_and_decide(base_votes, q, gt)
print(f"\n  Baseline: {b['final_answer']}  (GT: {gt})  {'CORRECT' if b['final_answer']==gt else 'WRONG'}")
print(f"  Vote counts: {b['vote_counts']}")
print("\nPipeline verified -- run Cell 12 for full evaluation")


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


SINGLE QUESTION TEST  (RACE-High)
Input (first 500 chars):
Passage:
In the dark and damp market,Chen ShuChu,near her sixties,owns a stall that her father left her.The stall,called YuanJin Vegetables,is her everything.Chen earns only a little money by selling vegetables,but she has donated about $321,550 to help poor children.
In March,Forbes magazine named her one of the 48 great philanthropists from the AsiaPacific region.A month later,TIME magazine selected the year's top 100 influential people and Chen was one of them.Although she has received many h

GT Answer: B

[1] Guide generating plan...
Plan:
Step 1: The passage mentions that "Although she has received many honours and rewards, Chen only cares about her vegetable stall and whether her regular customers buy her...". This indicates that while Chen receives recognition, her primary focus remains on her vegetable stall rather than external accolades.
Step 2: Looking at the options, we can see that none mention receiving awards or

In [12]:
# CELL 12 -- Full Dual Evaluation Loop

print(f"Dual evaluation: {len(test_data)} RACE-High questions")
print(f"Each question: {CONFIG['n_votes']} guided votes + {CONFIG['n_votes']} baseline votes")
print(f"Random baseline (chance): 25.0% (1 in 4 options)")
print("-" * 65)

all_results  = []
base_results = []
start_idx    = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        all_results  = [r for r in lines if r.get("mode") == "guided"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from index {start_idx}")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="RACE Eval"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    # ---- GUIDED -----------------------------------------------
    try:
        plan        = generate_plan(question)
        g_votes_raw = [extract_pred_answer(generate_guided(question, plan))
                       for _ in range(CONFIG["n_votes"])]
        g_dec       = vote_and_decide(g_votes_raw, question, gt_answer)

        all_results.append({
            "mode"             : "guided",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "plan"             : plan,
            "votes"            : g_votes_raw,
            "final_answer"     : g_dec["final_answer"],
            "correct"          : g_dec["final_answer"] == gt_answer,
            "strategy"         : g_dec["strategy"],
            "confidence"       : g_dec["confidence"],
            "vote_counts"      : g_dec["vote_counts"],
            "total_votes"      : g_dec["total_votes"],
            "correct_votes"    : g_dec["correct_votes"],
            "vote_consistency" : g_dec["vote_consistency"],
            "wasted_votes"     : g_dec["wasted_votes"],
            "refiner_used"     : g_dec["refiner_used"],
            "refiner_correct"  : g_dec["refiner_correct"],
        })
    except Exception as e:
        print(f"  [GUIDED ERROR idx={idx}]: {e}")
        all_results.append({"mode":"guided","idx":idx,"correct":False,
                             "gt_answer":gt_answer,"final_answer":"",
                             "strategy":"error","confidence":0.0,
                             "vote_consistency":0.0,"wasted_votes":5,
                             "refiner_used":False,"refiner_correct":None,
                             "vote_counts":{},"total_votes":5,"correct_votes":0})

    # ---- BASELINE ---------------------------------------------
    try:
        b_votes_raw = [extract_pred_answer(generate_baseline(question))
                       for _ in range(CONFIG["n_votes"])]
        b_dec       = vote_and_decide(b_votes_raw, question, gt_answer)

        base_results.append({
            "mode"             : "baseline",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "votes"            : b_votes_raw,
            "final_answer"     : b_dec["final_answer"],
            "correct"          : b_dec["final_answer"] == gt_answer,
            "strategy"         : b_dec["strategy"],
            "confidence"       : b_dec["confidence"],
            "vote_counts"      : b_dec["vote_counts"],
            "total_votes"      : b_dec["total_votes"],
            "correct_votes"    : b_dec["correct_votes"],
            "vote_consistency" : b_dec["vote_consistency"],
            "wasted_votes"     : b_dec["wasted_votes"],
            "refiner_used"     : b_dec["refiner_used"],
            "refiner_correct"  : b_dec["refiner_correct"],
        })
    except Exception as e:
        print(f"  [BASELINE ERROR idx={idx}]: {e}")
        base_results.append({"mode":"baseline","idx":idx,"correct":False,
                              "gt_answer":gt_answer,"final_answer":"",
                              "strategy":"error","confidence":0.0,
                              "vote_consistency":0.0,"wasted_votes":5,
                              "refiner_used":False,"refiner_correct":None,
                              "vote_counts":{},"total_votes":5,"correct_votes":0})

    # ---- Checkpoint -------------------------------------------
    if (idx + 1) % CONFIG["save_every"] == 0 or (idx + 1) == len(test_data):
        with open(CONFIG["results_file"], "w") as f:
            for r in all_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        elapsed = time.time() - t0
        g_acc_so_far = sum(r["correct"] for r in all_results) / len(all_results) * 100
         b_acc_so_far = sum(r["correct"] for r in base_results) / len(base_results) * 100
        print(f"  [{idx+1}/{len(test_data)}] Guided: {g_acc_so_far:.1f}%  "
              f"Baseline: {b_acc_so_far:.1f}%  ({elapsed/60:.1f}min)")

print("\n" + "=" * 65)
g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
print(f"  Guided   : {g_acc:.1f}%")
print(f"  Baseline : {b_acc:.1f}%")
print(f"  Delta    : +{g_acc - b_acc:.1f} pts")
print(f"  Random   : 25.0%")
print("=" * 65)


Dual evaluation: 500 RACE-High questions
Each question: 5 guided votes + 5 baseline votes
Random baseline (chance): 25.0% (1 in 4 options)
-----------------------------------------------------------------
Starting fresh


RACE Eval:   0%|          | 0/500 [00:00<?, ?it/s]

  [25/500] Guided: 68.0%  Baseline: 56.0%  (12.0min)
  [50/500] Guided: 70.0%  Baseline: 60.0%  (23.3min)
  [75/500] Guided: 68.0%  Baseline: 57.3%  (33.5min)
  [100/500] Guided: 61.0%  Baseline: 57.0%  (44.4min)
  [125/500] Guided: 62.4%  Baseline: 60.0%  (56.5min)
  [150/500] Guided: 62.7%  Baseline: 60.7%  (66.0min)
  [175/500] Guided: 62.9%  Baseline: 61.1%  (76.9min)
  [200/500] Guided: 61.5%  Baseline: 60.0%  (88.3min)
  [225/500] Guided: 63.1%  Baseline: 60.9%  (98.4min)
  [250/500] Guided: 62.4%  Baseline: 62.0%  (107.1min)
  [275/500] Guided: 62.2%  Baseline: 60.7%  (118.1min)
  [300/500] Guided: 61.0%  Baseline: 61.3%  (129.4min)
  [325/500] Guided: 59.7%  Baseline: 60.6%  (139.1min)
  [350/500] Guided: 58.6%  Baseline: 59.7%  (150.5min)
  [375/500] Guided: 58.9%  Baseline: 60.8%  (160.3min)
  [400/500] Guided: 59.0%  Baseline: 61.0%  (172.7min)
  [425/500] Guided: 59.1%  Baseline: 60.7%  (182.3min)
  [450/500] Guided: 58.0%  Baseline: 59.1%  (192.1min)
  [475/500] Guided: 57

In [13]:
# CELL 13 -- ANGLE 1: COMPUTE EFFICIENCY

G, S, N = CONFIG["guide_params_B"], CONFIG["solver_params_B"], CONFIG["n_votes"]
guided_compute   = (G * 1) + (S * N)
baseline_compute = S * N
upper_compute    = G * N
random_chance    = 25.0

g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
savings_pct = (1 - guided_compute / upper_compute) * 100

g_wasted = sum(r["wasted_votes"] for r in all_results)
b_wasted = sum(r["wasted_votes"] for r in base_results)

ref_triggered = sum(r["refiner_used"] for r in all_results)
ref_correct   = sum(1 for r in all_results if r["refiner_used"] and r.get("refiner_correct"))

strategy_stats = {}
for r in all_results:
    s = r["strategy"]
    if s not in strategy_stats: strategy_stats[s] = {"n":0,"correct":0}
    strategy_stats[s]["n"] += 1
    if r["correct"]: strategy_stats[s]["correct"] += 1

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (RACE-High)")
print("=" * 65)
print(f"  Random chance: {random_chance}% (4 options)")
print(f"\n  {'Setup':<32} | {'Compute':>10} | {'Accuracy':>9}")
print(f"  {'-'*32}-+-{'-'*10}-+-{'-'*9}")
print(f"  {'Random chance':<32} | {'--':>10} | {random_chance:>8.1f}%")
print(f"  {'Baseline (1.5B x5)':<32} | {baseline_compute:>8.1f}B  | {b_acc:>8.1f}%")
print(f"  {'Guided  (3B x1 + 1.5B x5)':<32} | {guided_compute:>8.1f}B  | {g_acc:>8.1f}%")
print(f"  {'Upper   (3B x5)':<32} | {upper_compute:>8.1f}B  | {'(ceiling)':>9}")
print(f"\n  Gain            : +{g_acc - b_acc:.1f} pts")
print(f"  Above chance    : +{g_acc - random_chance:.1f} pts")
print(f"  Compute savings : {savings_pct:.0f}% cheaper vs upper")
print(f"  Wasted saved    : {b_wasted - g_wasted}")
if ref_triggered:
    print(f"  Refiner: {ref_triggered} triggered, {ref_correct} correct ({ref_correct/ref_triggered*100:.1f}%)")

print("\n  Strategy breakdown:")
for strat, stats in strategy_stats.items():
    acc = stats["correct"] / stats["n"] * 100 if stats["n"] else 0
    print(f"    {strat:<20}: {stats['n']}q  |  {acc:.1f}% accurate")

a1_data = {
    "dataset": "RACE-High",
    "n_questions": len(all_results),
    "random_chance": random_chance,
    "guided_accuracy": round(g_acc, 1),
    "baseline_accuracy": round(b_acc, 1),
    "accuracy_gain": round(g_acc - b_acc, 1),
    "guided_above_chance": round(g_acc - random_chance, 1),
    "baseline_above_chance": round(b_acc - random_chance, 1),
    "guided_compute_B": guided_compute,
    "baseline_compute_B": baseline_compute,
    "upper_compute_B": upper_compute,
    "compute_savings_pct": round(savings_pct, 1),
    "guided_wasted_votes": g_wasted,
    "baseline_wasted_votes": b_wasted,
    "wasted_votes_saved": b_wasted - g_wasted,
    "refiner_triggered": ref_triggered,
    "refiner_correct": ref_correct,
    "strategy_stats": strategy_stats,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(a1_data, f, indent=2)
print(f"\nAngle 1 saved to {CONFIG['angle1_file']}")


ANGLE 1 -- COMPUTE EFFICIENCY  (RACE-High)
  Random chance: 25.0% (4 options)

  Setup                            |    Compute |  Accuracy
  ---------------------------------+------------+----------
  Random chance                    |         -- |     25.0%
  Baseline (1.5B x5)               |      7.5B  |     58.6%
  Guided  (3B x1 + 1.5B x5)        |     10.5B  |     57.4%
  Upper   (3B x5)                  |     15.0B  | (ceiling)

  Gain            : +-1.2 pts
  Above chance    : +32.4 pts
  Compute savings : 30% cheaper vs upper
  Wasted saved    : 44

  Strategy breakdown:
    majority            : 500q  |  57.4% accurate

Angle 1 saved to /kaggle/working/race_eval/angle1_compute_efficiency.json


In [14]:
# CELL 14 -- ANGLE 2: VOTE CONSISTENCY

g_cons = [r["vote_consistency"] for r in all_results]
b_cons = [r["vote_consistency"] for r in base_results]

g_mean = np.mean(g_cons)
b_mean = np.mean(b_cons)
lift   = g_mean / max(b_mean, 1e-6)

guided_wins   = sum(1 for g, b in zip(g_cons, b_cons) if g > b)
baseline_wins = sum(1 for g, b in zip(g_cons, b_cons) if b > g)
tied          = sum(1 for g, b in zip(g_cons, b_cons) if g == b)

def bucket(scores):
    return {
        "all_wrong  (0%)":  sum(1 for s in scores if s == 0.0),
        "low       (1-39%)":sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)": sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)": sum(1 for s in scores if s >= 0.8),
    }

g_dist = bucket(g_cons)
b_dist = bucket(b_cons)

all_letters_guided   = []
all_letters_baseline = []
for r in all_results:
    all_letters_guided.extend(r["vote_counts"].keys())
for r in base_results:
    all_letters_baseline.extend(r["vote_counts"].keys())
g_letter_dist = Counter(all_letters_guided)
b_letter_dist = Counter(all_letters_baseline)
total_g = sum(g_letter_dist.values())
total_b = sum(b_letter_dist.values())

all_wrong_guided   = sum(1 for r in all_results  if r["vote_consistency"] == 0.0)
all_wrong_baseline = sum(1 for r in base_results if r["vote_consistency"] == 0.0)

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (RACE-High)")
print("=" * 65)
print(f"\n  Mean correct-vote ratio:")
print(f"    Guided   : {g_mean*100:.1f}%  ({g_mean*CONFIG['n_votes']:.2f} / {CONFIG['n_votes']} votes correct avg)")
print(f"    Baseline : {b_mean*100:.1f}%  ({b_mean*CONFIG['n_votes']:.2f} / {CONFIG['n_votes']} votes correct avg)")
print(f"    Lift     : {lift:.2f}x")
print(f"\n  Per-question: Guided wins {guided_wins}, Baseline wins {baseline_wins}, Tied {tied}")
print(f"\n  {'Bucket':<22} | {'Guided':>8} | {'Baseline':>8} | {'Diff':>6}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}-+-{'-'*6}")
for bkt in ["all_wrong  (0%)", "low       (1-39%)", "medium  (40-79%)", "high   (80-100%)"]:
    gn, bn = g_dist[bkt], b_dist[bkt]
    print(f"  {bkt:<22} | {gn:>8} | {bn:>8} | {gn-bn:>+6}")

print(f"\n  Letter distribution (% of winning votes):")
print(f"  {'Letter':<8} | {'Guided':>8} | {'Baseline':>8}")
for ltr in sorted(VALID_LETTERS):
    gp = g_letter_dist.get(ltr, 0) / max(total_g, 1) * 100
    bp = b_letter_dist.get(ltr, 0) / max(total_b, 1) * 100
    print(f"  {ltr:<8} | {gp:>7.1f}% | {bp:>7.1f}%")

print(f"\n  All-wrong: Guided {all_wrong_guided}  |  Baseline {all_wrong_baseline}  |  Delta {all_wrong_baseline - all_wrong_guided} fewer")

a2_data = {
    "dataset": "RACE-High",
    "guided_mean_consistency": round(g_mean, 4),
    "baseline_mean_consistency": round(b_mean, 4),
    "consistency_lift": round(lift, 4),
    "guided_wins": guided_wins,
    "baseline_wins": baseline_wins,
    "tied": tied,
    "guided_distribution": g_dist,
    "baseline_distribution": b_dist,
    "all_wrong_guided": all_wrong_guided,
    "all_wrong_baseline": all_wrong_baseline,
    "guided_letter_dist": {k: round(v/max(total_g,1)*100,1) for k,v in g_letter_dist.items()},
    "baseline_letter_dist": {k: round(v/max(total_b,1)*100,1) for k,v in b_letter_dist.items()},
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(a2_data, f, indent=2)
print(f"\nAngle 2 saved to {CONFIG['angle2_file']}")


ANGLE 2 -- VOTE CONSISTENCY  (RACE-High)

  Mean correct-vote ratio:
    Guided   : 57.3%  (2.87 / 5 votes correct avg)
    Baseline : 58.1%  (2.90 / 5 votes correct avg)
    Lift     : 0.99x

  Per-question: Guided wins 78, Baseline wins 83, Tied 339

  Bucket                 |   Guided | Baseline |   Diff
  -----------------------+----------+----------+-------
  all_wrong  (0%)        |      203 |      188 |    +15
  low       (1-39%)      |        4 |       13 |     -9
  medium  (40-79%)       |       13 |       18 |     -5
  high   (80-100%)       |      280 |      281 |     -1

  Letter distribution (% of winning votes):
  Letter   |   Guided | Baseline
  A        |    21.8% |    14.2%
  B        |    20.7% |    25.8%
  C        |    28.1% |    36.8%
  D        |    27.9% |    23.3%

  All-wrong: Guided 203  |  Baseline 188  |  Delta -15 fewer

Angle 2 saved to /kaggle/working/race_eval/angle2_vote_consistency.json


In [15]:
# CELL 15 -- ANGLE 3: CONFIDENCE CALIBRATION

def calibration_report(results, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40, 0.25),
    ]
    n_total    = len(results)
    ece        = 0.0
    calib_out  = []
    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])

    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")
    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |")
            continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({"bucket":name,"count":n,"accuracy":round(acc,4),
                           "expected":mid,"gap":round(gap,4)})

    hc     = [r for r in results if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc) / max(1, len(hc)) * 100
    print(f"  ECE: {ece:.4f}")
    print(f"  High-conf: {len(hc)} questions  |  Accuracy: {hc_acc:.1f}%  |  Confidently WRONG: {false_conf}")
    return ece, calib_out, false_conf, hc_acc, len(hc)

print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (RACE-High)")
print("=" * 65)
g_ece, g_calib, g_fc, g_hc_acc, g_hc_n = calibration_report(all_results,  "GUIDED")
b_ece, b_calib, b_fc, b_hc_acc, b_hc_n = calibration_report(base_results, "BASELINE")

ece_imp = (b_ece - g_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE improvement : {ece_imp:.1f}% better calibrated with guidance")
print(f"  False confidence: Guided {g_fc}  vs  Baseline {b_fc}  ({b_fc - g_fc} fewer)")

a3_data = {
    "dataset": "RACE-High",
    "guided_ece": round(g_ece, 4),
    "baseline_ece": round(b_ece, 4),
    "ece_improvement_pct": round(ece_imp, 1),
    "guided_calibration": g_calib,
    "baseline_calibration": b_calib,
    "guided_false_confidence": g_fc,
    "baseline_false_confidence": b_fc,
    "guided_high_conf_accuracy": round(g_hc_acc, 1),
    "baseline_high_conf_accuracy": round(b_hc_acc, 1),
    "guided_high_conf_n": g_hc_n,
    "baseline_high_conf_n": b_hc_n,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(a3_data, f, indent=2)
print(f"\nAngle 3 saved to {CONFIG['angle3_file']}")


ANGLE 3 -- CONFIDENCE CALIBRATION  (RACE-High)

  [GUIDED]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        |   486 |     57.6% |       90% |  0.324 | Poor
  High       (0.60-0.80)     |    14 |     50.0% |       70% |  0.200 | Poor
  Medium     (0.40-0.60)     |    -- |        -- |       50% |     -- |
  Low        (<0.40)         |    -- |        -- |       25% |     -- |
  ECE: 0.3204
  High-conf: 486 questions  |  Accuracy: 57.6%  |  Confidently WRONG: 206

  [BASELINE]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        |   471 |     59.7% |       90% |  0.303 | Poor
  High       (0.60-0.80)     |    26 |     46.2% |       70% |  0.238 | Poor
  Medium     (0.40-0.60)     |     3 |      0.0% |       50% |  0.500 | Poor
  Lo

In [16]:
# CELL 16 -- Full Paper Summary

with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)

n = a1["n_questions"]
print("=" * 68)
print("  RACE-HIGH EVALUATION -- PAPER SUMMARY TABLE")
print("=" * 68)
print(f"  Dataset: RACE (high school)  |  N={n}  |  Seed={CONFIG['random_seed']}")
print(f"  Models : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver")
print(f"  Random chance: 25.0%  (4 options)")
print()

rows = [
    ["Metric", "Baseline", "Guided", "Change"],
    ["Overall Accuracy", str(a1["baseline_accuracy"])+"%", str(a1["guided_accuracy"])+"%",
     "+"+str(round(a1["guided_accuracy"]-a1["baseline_accuracy"],1))+" pts"],
    ["Above Random (25%)", "+"+str(a1["baseline_above_chance"])+" pts",
     "+"+str(a1["guided_above_chance"])+" pts", ""],
    ["Compute Cost", str(a1["baseline_compute_B"])+"B passes",
     str(a1["guided_compute_B"])+"B passes", str(a1["compute_savings_pct"])+"% cheaper vs ceiling"],
    ["Wasted Votes", str(a1["baseline_wasted_votes"]), str(a1["guided_wasted_votes"]),
     str(a1["wasted_votes_saved"])+" fewer"],
    ["Vote Consistency", str(round(a2["baseline_mean_consistency"]*100,1))+"%",
     str(round(a2["guided_mean_consistency"]*100,1))+"%", str(round(a2["consistency_lift"],2))+"x lift"],
    ["High-Agreement (80-100%)", str(a2["baseline_distribution"]["high   (80-100%)"]),
     str(a2["guided_distribution"]["high   (80-100%)"]), ""],
    ["All-Wrong (0/5)", str(a2["all_wrong_baseline"]), str(a2["all_wrong_guided"]),
     str(a2["all_wrong_baseline"]-a2["all_wrong_guided"])+" fewer"],
    ["ECE (lower=better)", str(a3["baseline_ece"]), str(a3["guided_ece"]),
     str(a3["ece_improvement_pct"])+"% better"],
    ["High-Conf Accuracy (>=0.80)",
     str(a3["baseline_high_conf_accuracy"])+"% (n="+str(a3["baseline_high_conf_n"])+")",
     str(a3["guided_high_conf_accuracy"])+"% (n="+str(a3["guided_high_conf_n"])+")", ""],
    ["False Confidence", str(a3["baseline_false_confidence"]), str(a3["guided_false_confidence"]),
     str(a3["baseline_false_confidence"]-a3["guided_false_confidence"])+" fewer"],
]

col_w = [32, 26, 26, 30]
print("  "+" | ".join(f"{rows[0][i]:<{col_w[i]}}" for i in range(4)))
print("  "+"+-".join("-"*w for w in col_w))
for row in rows[1:]:
    print("  "+" | ".join(f"{str(row[i]):<{col_w[i]}}" for i in range(4)))

print()
print(f"  KEY FINDING: +{a1['accuracy_gain']:.1f} pts on reading comprehension + inference")
print(f"  at 30% lower compute than ceiling. ECE improves {a3['ece_improvement_pct']:.1f}%.")
print("=" * 68)


  RACE-HIGH EVALUATION -- PAPER SUMMARY TABLE
  Dataset: RACE (high school)  |  N=500  |  Seed=42
  Models : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver
  Random chance: 25.0%  (4 options)

  Metric                           | Baseline                   | Guided                     | Change                        
  --------------------------------+---------------------------+---------------------------+-------------------------------
  Overall Accuracy                 | 58.6%                      | 57.4%                      | +-1.2 pts                     
  Above Random (25%)               | +33.6 pts                  | +32.4 pts                  |                               
  Compute Cost                     | 7.5B passes                | 10.5B passes               | 30.0% cheaper vs ceiling      
  Wasted Votes                     | 88                         | 44                         | 44 fewer                      
  Vote Consistency                 | 58.1%                  